# Training ConvNeXtTiny untuk Aplikasi Cek Penulis

ConvNeXtTiny dipilih karena akurasinya paling tinggi di eksperimen paper (98,05%).
Notebook ini melatih ulang ConvNeXtTiny dengan pengaturan yang sama, lalu menyimpan modelnya dalam format `.keras` untuk aplikasi Streamlit.

Langkah menjalankan:
1. Runtime → Change runtime type → **T4 GPU**
2. Pastikan zip dataset ada di `MyDrive/MLA/`
3. Runtime → **Run all**, izinkan akses Google Drive

Hasil disimpan di `MyDrive/Model_CekPenulis`. Yang dipakai aplikasi adalah `model_cek_penulis.zip` (isinya folder `model` dan `contoh_uji`).

## Pengaturan

In [ ]:
# zip dataset di Drive dan folder tempat hasil training disimpan
ZIP_DATASET = '/content/drive/MyDrive/MLA/Dataset_Tulis_Tangan_Mahasiswa.zip'
FOLDER_SIMPAN = '/content/drive/MyDrive/Model_CekPenulis'

# ukuran input dan ambang harus sama dengan utils/config.py di aplikasi
TINGGI = 224
LEBAR = 672
AMBANG_DIKENALI = 0.5

# sama dengan eksperimen paper, kalau OOM ganti BATCH jadi 8
BATCH = 16
EPOCH = 15
MIXED_PRECISION = True

## Cek GPU

In [ ]:
import os
import sys
import shutil
import subprocess

# jalanin python di proses terpisah, ambil baris terakhir outputnya
def jalankan_python(perintah):
    hasil = subprocess.run([sys.executable, '-c', perintah], capture_output=True, text=True)
    baris = hasil.stdout.strip().splitlines()
    return baris[-1] if baris else ''

def jumlah_gpu():
    hasil = jalankan_python('import tensorflow as tf; print(len(tf.config.list_physical_devices("GPU")))')
    return int(hasil) if hasil.isdigit() else 0

# cek GPU Colab sudah nyala atau belum
if shutil.which('nvidia-smi') is None or subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise RuntimeError('GPU belum tersambung. Pilih Runtime > Change runtime type > T4 GPU, lalu Run all lagi.')

# GPU ada tapi belum kebaca TensorFlow, install versi CUDA-nya
if jumlah_gpu() == 0:
    versi = jalankan_python('import tensorflow as tf; print(tf.__version__)')
    paket = f'tensorflow[and-cuda]=={versi}' if versi else 'tensorflow[and-cuda]'
    print('TensorFlow belum membaca GPU, install', paket)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', paket])
    if jumlah_gpu() == 0:
        raise RuntimeError('TensorFlow tetap tidak membaca GPU. Pilih Runtime > Disconnect and delete runtime, lalu Run all lagi.')

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)

## Import library

In [ ]:
import os
import gc
import glob
import json
import time
import random
import shutil
import hashlib
import zipfile

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from google.colab import drive

# seed tetap biar hasil bisa diulang
keras.utils.set_random_seed(42)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Ambil dataset dari Google Drive

In [ ]:
# sambungkan ke Drive
drive.mount('/content/drive')

# kalau path salah, cari otomatis di Drive
if not os.path.exists(ZIP_DATASET):
    kandidat = sorted(glob.glob('/content/drive/MyDrive/**/Dataset_Tulis_Tangan_Mahasiswa*.zip', recursive=True))
    if not kandidat:
        raise FileNotFoundError('Zip dataset tidak ditemukan di Google Drive')
    ZIP_DATASET = kandidat[0]

# ekstrak dataset
shutil.rmtree('/content/data', ignore_errors=True)
with zipfile.ZipFile(ZIP_DATASET) as z:
    z.extractall('/content/data')

# folder dataset = folder yang isinya >= 10 subfolder (1 folder = 1 penulis)
FOLDER_DATASET = None
for root, dirs, files in os.walk('/content/data'):
    if len(dirs) >= 10:
        FOLDER_DATASET = root
        break

# folder kerja sementara, nanti disalin ke Drive di akhir
FOLDER_KERJA = '/content/hasil'
FOLDER_MODEL = f'{FOLDER_KERJA}/model'
FOLDER_CONTOH = f'{FOLDER_KERJA}/contoh_uji'
shutil.rmtree(FOLDER_KERJA, ignore_errors=True)
os.makedirs(FOLDER_MODEL)
os.makedirs(FOLDER_CONTOH)
print('Zip     :', ZIP_DATASET)
print('Dataset :', FOLDER_DATASET)

## Bagi data train, validation, test

Sama dengan notebook paper: 70/15/15 per penulis, seed 42, gambar yang isinya sama persis hanya dipakai sekali.

In [ ]:
EKSTENSI = ('.jpg', '.jpeg', '.png')
# nama folder = NIM penulis = kelas
kelas = sorted(d for d in os.listdir(FOLDER_DATASET) if os.path.isdir(os.path.join(FOLDER_DATASET, d)))

random.seed(42)
hash_terpakai = {}
duplikat = []
data = {'train': [], 'validation': [], 'test': []}

# tiap penulis: buang duplikat (cek md5), acak, ambil 15% test, 15% val, sisanya train
for i, nama in enumerate(kelas):
    folder = os.path.join(FOLDER_DATASET, nama)
    unik = []
    for f in sorted(os.listdir(folder)):
        if not f.lower().endswith(EKSTENSI):
            continue
        path = os.path.join(folder, f)
        h = hashlib.md5(open(path, 'rb').read()).hexdigest()
        if h in hash_terpakai:
            duplikat.append(f'{nama}/{f}')
            continue
        hash_terpakai[h] = path
        unik.append(path)

    random.shuffle(unik)
    n_test = max(1, round(len(unik) * 0.15))
    n_val = max(1, round(len(unik) * 0.15))
    data['test'] += [(p, i) for p in unik[:n_test]]
    data['validation'] += [(p, i) for p in unik[n_test:n_test + n_val]]
    data['train'] += [(p, i) for p in unik[n_test + n_val:]]

n_kelas = len(kelas)
print('Jumlah penulis :', n_kelas)
for bagian in data:
    print(f'{bagian:10s} : {len(data[bagian])} citra')
print('Duplikat dibuang:', len(duplikat))

## Preprocessing

Fungsinya sama persis dengan `utils/preprocess.py` di aplikasi: grayscale, ratakan cahaya, crop area tulisan, warna dibalik, lalu diperkecil ke 224 × 672 tanpa merusak rasio.

In [ ]:
# buka gambar, putar sesuai EXIF, jadi grayscale, yang terlalu besar diperkecil
def buka_gambar(path):
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
    gray = np.array(img.convert('L'))
    if max(gray.shape) > 2000:
        skala = 2000 / max(gray.shape)
        gray = cv2.resize(gray, None, fx=skala, fy=skala, interpolation=cv2.INTER_AREA)
    return gray

# latar kertas ditebak pakai median blur lalu dibagi, jadi bayangan foto hilang
def ratakan_cahaya(gray):
    latar = cv2.medianBlur(gray, 31)
    return cv2.divide(gray, latar, scale=255)

# cari kotak area yang ada tintanya, sisakan margin 4 piksel
def cari_area_tulisan(gray):
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    tinta = mask > 0
    h, w = gray.shape
    baris = np.where(tinta.sum(axis=1) > max(2, 0.002 * w))[0]
    kolom = np.where(tinta.sum(axis=0) > max(2, 0.002 * h))[0]
    if len(baris) == 0 or len(kolom) == 0:
        return 0, h, 0, w
    y1, y2 = max(0, baris[0] - 4), min(h, baris[-1] + 5)
    x1, x2 = max(0, kolom[0] - 4), min(w, kolom[-1] + 5)
    return y1, y2, x1, x2

# balik warna, kecilkan sesuai rasio, taruh di tengah kanvas hitam
def siapkan_input(crop):
    h, w = crop.shape
    skala = min(TINGGI / h, LEBAR / w)
    th, tw = max(1, round(h * skala)), max(1, round(w * skala))
    cara = cv2.INTER_AREA if skala < 1 else cv2.INTER_LINEAR
    kecil = cv2.resize(255 - crop, (tw, th), interpolation=cara)
    kanvas = np.zeros((TINGGI, LEBAR), dtype=np.uint8)
    atas, kiri = (TINGGI - th) // 2, (LEBAR - tw) // 2
    kanvas[atas:atas + th, kiri:kiri + tw] = kecil
    return kanvas

def proses_gambar(path):
    gray = ratakan_cahaya(buka_gambar(path))
    y1, y2, x1, x2 = cari_area_tulisan(gray)
    return siapkan_input(gray[y1:y2, x1:x2])

# proses semua gambar di satu bagian jadi array
def muat(bagian):
    X = np.array([proses_gambar(p) for p, _ in data[bagian]])[..., None]
    y = np.array([label for _, label in data[bagian]])
    return X, y

mulai = time.time()
X_train, y_train = muat('train')
X_val, y_val = muat('validation')
X_test, y_test = muat('test')
print(f'Preprocessing selesai dalam {time.time() - mulai:.1f} detik')
print('Train', X_train.shape, '| Validation', X_val.shape, '| Test', X_test.shape)

# tampilkan 4 contoh hasilnya
plt.figure(figsize=(14, 6))
for i, j in enumerate(np.linspace(0, len(X_train) - 1, 4).astype(int)):
    plt.subplot(4, 1, i + 1)
    plt.imshow(X_train[j, :, :, 0], cmap='gray')
    plt.title(kelas[y_train[j]], fontsize=10)
    plt.axis('off')
plt.tight_layout()
plt.show()

## Pipeline data dan augmentasi

In [ ]:
# augmentasi ringan: rotasi, zoom, kecerahan
augmentasi = keras.Sequential([
    layers.RandomRotation(0.01, fill_mode='constant'),
    layers.RandomZoom(0.1, fill_mode='constant'),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
])

# bobot pretrained butuh 3 channel, jadi grayscale diduplikasi ke RGB
def ke_rgb(x, y):
    return tf.image.grayscale_to_rgb(tf.cast(x, tf.float32)), y

# augmentasi cuma untuk data latih
def buat_dataset(X, y, latih=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if latih:
        ds = ds.shuffle(len(X), seed=42)
    ds = ds.batch(BATCH).map(ke_rgb, num_parallel_calls=tf.data.AUTOTUNE)
    if latih:
        ds = ds.map(lambda a, b: (augmentasi(a, training=True), b), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

ds_train = buat_dataset(X_train, y_train, latih=True)
ds_val = buat_dataset(X_val, y_val)
ds_test = buat_dataset(X_test, y_test)

## Membangun model ConvNeXtTiny

Backbone ConvNeXtTiny (bobot ImageNet) + Global Average Pooling + Dropout 0.3 + Dense softmax.
Nama layer `gap` dan `prediksi` jangan diganti, karena dipakai Grad-CAM di aplikasi.

In [ ]:
# preprocessing bawaan ConvNeXt sudah ada di dalam model, jadi input tetap bernilai 0-255
def buat_model(bobot='imagenet'):
    inp = keras.Input(shape=(TINGGI, LEBAR, 3))
    base = keras.applications.ConvNeXtTiny(include_top=False, weights=bobot, input_shape=(TINGGI, LEBAR, 3))
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.3, name='dropout')(x)
    out = layers.Dense(n_kelas, activation='softmax', dtype='float32', name='prediksi')(x)
    return keras.Model(inp, out, name='cek_penulis_convnexttiny')

## Training

Aturannya sama dengan eksperimen paper: Adam 1e-4, early stopping (patience 4), learning rate diturunkan kalau val_loss tidak membaik.
Kalau mixed precision bikin loss NaN, training otomatis diulang dengan float32.

In [ ]:
def latih(policy):
    keras.backend.clear_session()
    keras.mixed_precision.set_global_policy(policy)
    keras.utils.set_random_seed(42)
    model = buat_model()
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    callback = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1),
        keras.callbacks.TerminateOnNaN(),
    ]
    mulai = time.time()
    hist = model.fit(ds_train, validation_data=ds_val, epochs=EPOCH, callbacks=callback, verbose=1)
    return model, hist.history, time.time() - mulai

policy = 'mixed_float16' if MIXED_PRECISION else 'float32'
model, riwayat, waktu_latih = latih(policy)
if np.isnan(riwayat['loss']).any() and policy != 'float32':
    print('Loss NaN, training diulang dengan float32')
    model, riwayat, waktu_latih = latih('float32')

# bobot yang dipakai = epoch dengan val_loss terkecil (restore_best_weights)
epoch_terbaik = int(np.argmin(riwayat['val_loss'])) + 1
print(f'Training selesai dalam {waktu_latih / 60:.1f} menit, epoch terbaik: {epoch_terbaik}')

## Grafik training

In [ ]:
# akurasi dan loss data latih vs validasi per epoch
epoch = range(1, len(riwayat['loss']) + 1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(epoch, riwayat['accuracy'], marker='o', label='Train')
ax[0].plot(epoch, riwayat['val_accuracy'], marker='o', label='Validation')
ax[0].set_title('Accuracy')
ax[1].plot(epoch, riwayat['loss'], marker='o', label='Train')
ax[1].plot(epoch, riwayat['val_loss'], marker='o', label='Validation')
ax[1].set_title('Loss')
for a in ax:
    a.axvline(epoch_terbaik, color='gray', linestyle='--', linewidth=1)
    a.set_xlabel('Epoch')
    a.legend()
    a.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FOLDER_MODEL}/grafik_training.png', dpi=130)
plt.show()

## Siapkan model untuk aplikasi

Model dilatih pakai mixed precision (float16). Aplikasi jalan di laptop (CPU), jadi bobotnya dipindah ke model float32 dengan arsitektur yang sama, tanpa optimizer.
Hasilnya file `.keras` lebih kecil dan prediksi di CPU lebih lancar. Bobotnya tetap sama persis.

In [ ]:
keras.mixed_precision.set_global_policy('float32')
model_aplikasi = buat_model(bobot=None)
model_aplikasi.set_weights(model.get_weights())
print('Parameter:', f'{model_aplikasi.count_params() / 1e6:.2f} juta')

## Evaluasi di data test

In [ ]:
# prediksi data test pakai model yang nanti dipakai aplikasi
prob = model_aplikasi.predict(ds_test, verbose=0)
pred = prob.argmax(axis=1)
conf = prob.max(axis=1)

# metrik macro (rata-rata semua penulis)
hasil_uji = {
    'accuracy': accuracy_score(y_test, pred),
    'precision': precision_score(y_test, pred, average='macro', zero_division=0),
    'recall': recall_score(y_test, pred, average='macro', zero_division=0),
    'f1_score': f1_score(y_test, pred, average='macro', zero_division=0),
}
for k, v in hasil_uji.items():
    print(f'{k:10s}: {v * 100:.2f}%')

# laporan per penulis disimpan ke file
laporan = classification_report(y_test, pred, labels=range(n_kelas), target_names=kelas, digits=4, zero_division=0)
open(f'{FOLDER_MODEL}/classification_report.txt', 'w').write(laporan)

# pengaruh ambang "Tidak Dikenali" ke jumlah gambar yang dikenali dan akurasinya
tabel_ambang = []
for a in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    lolos = conf >= a
    benar = (pred[lolos] == y_test[lolos]).mean() if lolos.any() else 0
    tabel_ambang.append({'Ambang': a, 'Dikenali (%)': lolos.mean() * 100, 'Akurasi saat dikenali (%)': benar * 100})
pd.DataFrame(tabel_ambang).round(2)

## Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, pred, labels=range(n_kelas))
plt.figure(figsize=(15, 13))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(n_kelas), kelas, rotation=90, fontsize=7)
plt.yticks(range(n_kelas), kelas, fontsize=7)
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix ConvNeXtTiny')
plt.tight_layout()
plt.savefig(f'{FOLDER_MODEL}/confusion_matrix.png', dpi=130)
plt.show()

# penulis yang paling sering salah ditebak
salah = cm.sum(axis=1) - np.diag(cm)
for i in np.argsort(salah)[::-1][:5]:
    if salah[i] > 0:
        print(f'{kelas[i]}: {salah[i]} dari {cm[i].sum()} citra salah')

## Simpan model dan file pendukung

File yang dibutuhkan aplikasi: `penulis_convnexttiny.keras`, `kelas.json`, `metrics.json`, `ringkasan_split.json`, ditambah grafik untuk halaman Tentang Model.

In [ ]:
model_aplikasi.save(f'{FOLDER_MODEL}/penulis_convnexttiny.keras')

# urutan kelas harus sama dengan urutan output model
json.dump(kelas, open(f'{FOLDER_MODEL}/kelas.json', 'w'), indent=2)

# jumlah data per bagian, ditampilkan di halaman Tentang Model
ringkasan = {'jumlah_penulis': n_kelas, 'total': {k: len(v) for k, v in data.items()}, 'duplikat': duplikat, 'seed': 42}
json.dump(ringkasan, open(f'{FOLDER_MODEL}/ringkasan_split.json', 'w'), indent=2)

# metrik test + info training
lolos = conf >= AMBANG_DIKENALI
metrik = {k: float(v) for k, v in hasil_uji.items()}
metrik.update({
    'model': 'ConvNeXtTiny',
    'jumlah_test': int(len(y_test)),
    'ambang': AMBANG_DIKENALI,
    'persen_dikenali': float(lolos.mean()),
    'akurasi_saat_dikenali': float((pred[lolos] == y_test[lolos]).mean()) if lolos.any() else 0.0,
    'epoch': len(riwayat['loss']),
    'epoch_terbaik': epoch_terbaik,
    'waktu_latih_menit': round(waktu_latih / 60, 2),
    'parameter_juta': round(model_aplikasi.count_params() / 1e6, 2),
    'ukuran_input': [TINGGI, LEBAR],
    'tensorflow': tf.__version__,
    'keras': keras.__version__,
})
json.dump(metrik, open(f'{FOLDER_MODEL}/metrics.json', 'w'), indent=2)
json.dump({k: [float(v) for v in vals] for k, vals in riwayat.items()}, open(f'{FOLDER_MODEL}/riwayat_training.json', 'w'), indent=2)
print(json.dumps(metrik, indent=2))

## Cek file model bisa dibuka lagi

In [ ]:
# buka lagi file .keras, hasil prediksinya harus sama dengan model sebelum disimpan
path_model = f'{FOLDER_MODEL}/penulis_convnexttiny.keras'
cek = keras.models.load_model(path_model, compile=False)
x_cek = np.repeat(X_test[:8], 3, axis=-1).astype('float32')
selisih = np.abs(cek.predict(x_cek, verbose=0) - model_aplikasi.predict(x_cek, verbose=0)).max()
print('Selisih prediksi :', selisih)
print('Ukuran file      :', round(os.path.getsize(path_model) / 1024 / 1024, 1), 'MB')
print('Layer            :', [l.name for l in cek.layers])

## Contoh gambar untuk mencoba aplikasi

Diambil 1 gambar per penulis dari data test, jadi gambar ini tidak pernah dilihat model saat training.

In [ ]:
# nama file diawali NIM penulis aslinya, biar gampang dicocokkan dengan hasil aplikasi
sudah = set()
for path, label in data['test']:
    if label in sudah:
        continue
    sudah.add(label)
    shutil.copy(path, f'{FOLDER_CONTOH}/{kelas[label]}_{os.path.basename(path)}')
print(len(os.listdir(FOLDER_CONTOH)), 'contoh gambar disalin')

## Simpan ke Google Drive

In [ ]:
# zip folder model dan contoh_uji jadi satu, lalu salin semuanya ke Drive
shutil.make_archive('/content/model_cek_penulis', 'zip', FOLDER_KERJA)
os.makedirs(FOLDER_SIMPAN, exist_ok=True)
shutil.copytree(FOLDER_KERJA, FOLDER_SIMPAN, dirs_exist_ok=True)
shutil.copy('/content/model_cek_penulis.zip', FOLDER_SIMPAN)

print('Tersimpan di', FOLDER_SIMPAN)
for f in sorted(os.listdir(FOLDER_SIMPAN)):
    print(' ', f)
print('Isi folder model:')
for f in sorted(os.listdir(f'{FOLDER_SIMPAN}/model')):
    print('   ', f)